# C11_E01 — Discretización Tustin/ZOH

**Capítulo:** 11 — Control digital, muestreo e implementación en PLC/DCS  
**Identificador:** `C11_E01`  
**Objetivo pedagógico:** Discretizar con criterio y verificar que la respuesta se preserva.  
**Librerías:** python-control, numpy, matplotlib

## Ejemplo industrial

PID continuo que se va a implementar en PLC con periodo de muestreo realista.

---

> **Aviso de uso responsable.** Este notebook es didáctico. No es código de producción. Cualquier implementación real requiere validación independiente. Ver `docs/politica_uso_responsable.md`.

## 1. Discretización del lazo PI

In [1]:
import control as ct
import numpy as np
import matplotlib.pyplot as plt
import os

G = ct.tf([1.0], [3.0, 1.0])
PI = ct.tf([1.8*3.0, 1.8], [3.0, 0.0])
Lc = PI*G

print("Polos continuos:", ct.poles(Lc))
print()
for method in ["tustin", "zoh"]:
    Ld = ct.c2d(Lc, Ts=0.2, method=method)
    print(f"Método {method:6}  →  polos discretos: {ct.poles(Ld)}")

Polos continuos: [-0.33333333+0.j  0.        +0.j]

Método tustin  →  polos discretos: [1.        +0.j 0.93548387+0.j]
Método zoh     →  polos discretos: [1.        +0.j 0.93550699+0.j]


## 2. Comparativa de respuesta cerrada continua vs discreta

In [2]:
# Lazo cerrado continuo
T_cont = ct.feedback(Lc, 1)
t = np.linspace(0, 30, 600)
_, y_c = ct.step_response(T_cont, t)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y_c, label="Continuo")
for method in ["tustin", "zoh"]:
    Ld = ct.c2d(Lc, Ts=0.2, method=method)
    Td = ct.feedback(Ld, 1)
    t_d = np.arange(0, 30, 0.2)
    _, y_d = ct.step_response(Td, t_d)
    ax.step(t_d, y_d, where='post', label=f"Discreto {method} (Ts=0.2)", linestyle='--')
ax.axhline(1.0, ls=':', color='gray')
ax.set_xlabel("t (s)"); ax.set_ylabel("y/SP"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("C11_E01 — Respuesta continua vs discreta")
out = '../../figures/cap11/fig_C11_F01_continuo_vs_discreto.png'
os.makedirs(os.path.dirname(out), exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=120); plt.show()

## 3. Interpretación

Con `Ts = 0.2 s ≈ 0.07·τ`, las dos discretizaciones (Tustin y ZOH) producen respuestas casi indistinguibles del continuo. Aumentar `Ts` por encima de `0.1·τ` empieza a degradar el margen de fase. **Discretizar un PID afinado en continuo no garantiza un PID discreto bueno** si `Ts` no se respeta.